In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# MAGFiLO 1.0: Solar Filament Segmentation (Final Submission)

## 1. Overview
This notebook reproduces our final submission for the MAGFiLO 1.0 competition (Public LB 0.39).

**Pipeline Architecture:**
- **Detector:** YOLO11m-seg.
- **Refiners:** EfficientNet-B3, EfficientNet-B4, and ResNet18-UNet "Fus".
- **Multi-Scale:** 5 scales (224–384) with probability averaging.
- **Post-processing:** `subtract_overlaps` to ensure zero overlapping masks.

**Weights:**
All model weights are publicly available in the attached Kaggle Dataset `magfilo-final-weights-all`.

In [27]:
# 1. Setup & Load Models

!pip install -q ultralytics segmentation-models-pytorch pycocotools

import os, glob, cv2, csv, numpy as np, torch, torch.nn as nn
from collections import defaultdict
from tqdm import tqdm
import segmentation_models_pytorch as smp
from torchvision.models import resnet18
from pycocotools import mask as mu
from ultralytics import YOLO

# Paths (all local)
COMP = glob.glob('/kaggle/input/**/MAGFiLO_1.0_Kaggle_2026', recursive=True)[0]
YOLO_PT = glob.glob('/kaggle/input/**/yolo11m_max_best.pt', recursive=True)[0]
REFINER_B3 = glob.glob('/kaggle/input/**/our_refiner_v3_efficientnet_b3.pth', recursive=True)[0]
REFINER_B4 = glob.glob('/kaggle/input/**/our_refiner_v3_efficientnet_b4.pth', recursive=True)[0]
FUSUSU_PT = glob.glob('/kaggle/input/**/final_refiner.pt', recursive=True)[0]

DEVICE = 'cuda'
H, W = 2048, 2048
IMG_SIZE = 1280
SCALES = [224, 256, 288, 320, 384]
CROP_CONTEXT = 1.8
MEAN = torch.tensor([0.485,0.456,0.406]).view(1,3,1,1).to(DEVICE)
STD  = torch.tensor([0.229,0.224,0.225]).view(1,3,1,1).to(DEVICE)
MIN_AREA_DET = 32
MIN_AREA_FINAL = 200
YOLO_CONF = 0.30
REF_THR = 0.60

class ConvBlock(nn.Sequential):
    def __init__(self, i, o):
        super().__init__(
            nn.Conv2d(i,o,3,padding=1,bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
            nn.Conv2d(o,o,3,padding=1,bias=False), nn.BatchNorm2d(o), nn.ReLU(inplace=True))

class Fus(nn.Module):
    def __init__(self):
        super().__init__()
        e = resnet18(weights=None)
        self.stem = nn.Sequential(e.conv1, e.bn1, e.relu); self.pool = e.maxpool
        self.layer1, self.layer2, self.layer3, self.layer4 = e.layer1, e.layer2, e.layer3, e.layer4
        self.up4 = nn.ConvTranspose2d(512,256,2,2); self.dec4 = ConvBlock(512,256)
        self.up3 = nn.ConvTranspose2d(256,128,2,2); self.dec3 = ConvBlock(256,128)
        self.up2 = nn.ConvTranspose2d(128,64,2,2);  self.dec2 = ConvBlock(128,64)
        self.up1 = nn.ConvTranspose2d(64,32,2,2);   self.dec1 = ConvBlock(96,32)
        self.up0 = nn.ConvTranspose2d(32,16,2,2);   self.head = nn.Conv2d(16,1,1)
        self.register_buffer("mean", torch.tensor([0.485,0.456,0.406]).view(1,3,1,1))
        self.register_buffer("std",  torch.tensor([0.229,0.224,0.225]).view(1,3,1,1))
    def forward(self, x):
        x = (x - self.mean) / self.std
        x0 = self.stem(x); x1 = self.layer1(self.pool(x0))
        x2 = self.layer2(x1); x3 = self.layer3(x2); x4 = self.layer4(x3)
        v = self.dec4(torch.cat([self.up4(x4), x3], 1))
        v = self.dec3(torch.cat([self.up3(v), x2], 1))
        v = self.dec2(torch.cat([self.up2(v), x1], 1))
        v = self.dec1(torch.cat([self.up1(v), x0], 1))
        return self.head(self.up0(v))

print("Loading models...")
refiners = []
b3 = smp.Unet('efficientnet-b3', encoder_weights=None, in_channels=3, classes=1).to(DEVICE)
b3.load_state_dict(torch.load(REFINER_B3, map_location=DEVICE))
b3.eval(); refiners.append(('b3', b3, True))

b4 = smp.Unet('efficientnet-b4', encoder_weights=None, in_channels=3, classes=1).to(DEVICE)
b4.load_state_dict(torch.load(REFINER_B4, map_location=DEVICE))
b4.eval(); refiners.append(('b4', b4, True))

fus = Fus().to(DEVICE)
fus.load_state_dict(torch.load(FUSUSU_PT, map_location='cpu', weights_only=False)['model'], strict=True)
fus.eval(); refiners.append(('fus', fus, False))

yolo = YOLO(YOLO_PT)
print("All models loaded.")

Loading models...
All models loaded.


## 2. Preprocessing & Inference
The input images (2048x2048) are resized to 1280x1280 for the YOLO detector. For each detected mask, a square crop is extracted and passed through the three refiners at 5 different scales. 
Probabilities are averaged and thresholded at 0.60.

In [28]:
# 2. Inference Pipeline

def square_bounds(mask, context=CROP_CONTEXT, minimum=96):
    ys, xs = np.where(mask)
    if len(ys) == 0: return None
    cx = (xs.min()+xs.max()+1)/2; cy = (ys.min()+ys.max()+1)/2
    side = max(int(np.ceil(max(xs.max()-xs.min()+1, ys.max()-ys.min()+1)*context)), minimum)
    side = min(side, min(H, W))
    x0 = int(round(cx-side/2)); y0 = int(round(cy-side/2))
    x1, y1 = x0+side, y0+side
    if x0<0: x1-=x0; x0=0
    if y0<0: y1-=y0; y0=0
    if x1>W: x0-=x1-W; x1=W
    if y1>H: y0-=y1-H; y1=H
    return int(x0),int(y0),int(x1),int(y1)

def encode(m):
    r = mu.encode(np.asfortranarray(m.astype(np.uint8)))
    c = r['counts']
    return c.decode('ascii') if isinstance(c, bytes) else str(c)

def subtract_overlaps(masks, min_area=MIN_AREA_FINAL):
    masks = sorted(masks, key=lambda m: -int(m.sum()))
    out = []
    for m in masks:
        for prev in out:
            m = np.logical_and(m, np.logical_not(prev))
        if m.sum() >= min_area: out.append(m)
    return out

@torch.no_grad()
def infer_image_yolo(image, conf_thr=YOLO_CONF, ref_thr=REF_THR):
    resized = cv2.resize(image, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
    resized_rgb = cv2.cvtColor(resized, cv2.COLOR_GRAY2RGB)
    r = yolo.predict(resized_rgb, conf=conf_thr, iou=0.5, verbose=False, device=0)[0]
    if r.masks is None: return []
    masks_yolo = r.masks.data.cpu().numpy()
    scores_yolo = r.boxes.conf.cpu().numpy()
    tasks = []
    for dm, sc in zip(masks_yolo, scores_yolo):
        dm_bin = (dm >= 0.5).astype(np.uint8)
        if dm_bin.sum() < MIN_AREA_DET: continue
        full = cv2.resize(dm_bin, (W, H), interpolation=cv2.INTER_NEAREST).astype(bool)
        b = square_bounds(full)
        if b is None: continue
        x0,y0,x1,y1 = b
        tasks.append({'bounds': b, 'crop': image[y0:y1, x0:x1],
                      'fallback': full, 'score': float(sc)})
    results = []
    for task in tasks:
        x0,y0,x1,y1 = task['bounds']
        probs = np.zeros((256,256), np.float32); n = 0
        for sc_ in SCALES:
            cs = cv2.resize(task['crop'], (sc_, sc_), interpolation=cv2.INTER_LINEAR)
            batch = torch.from_numpy(cs).float().unsqueeze(0).unsqueeze(0).repeat(1,3,1,1).to(DEVICE)/255.
            for name, ref, need_norm in refiners:
                x = (batch - MEAN)/STD if need_norm else batch
                p = torch.sigmoid(ref(x))[0,0].cpu().numpy()
                probs += cv2.resize(p, (256,256), interpolation=cv2.INTER_LINEAR); n += 1
        probs /= n
        cm = cv2.resize(probs, (x1-x0, y1-y0), interpolation=cv2.INTER_LINEAR) >= ref_thr
        if not cm.any():
            full = task['fallback'].copy()
        else:
            full = np.zeros((H,W), bool); full[y0:y1, x0:x1] = cm
        if full.sum() < MIN_AREA_FINAL: continue
        results.append((task['score'], full))
    return results

test_files = sorted(glob.glob(f'{COMP}/test/test_images/*.jpeg'))
all_masks = {}
for fpath in tqdm(test_files, desc='Inference'):
    fname = os.path.basename(fpath).replace('.jpeg', '')
    img = cv2.imread(fpath, cv2.IMREAD_GRAYSCALE)
    if img is None: continue
    cands = infer_image_yolo(img, conf_thr=YOLO_CONF, ref_thr=REF_THR)
    if cands: all_masks[fname] = cands

print(f"Inference complete. Processed {len(all_masks)} images.")

Inference: 100%|██████████| 180/180 [06:23<00:00,  2.13s/it]

Inference complete. Processed 178 images.


## 3. Post-processing & Submission
After inference, we resolve overlapping masks by sorting them by area and subtracting intersections. Finally, we encode the masks in RLE format and save the submission CSV.

In [30]:
# 3. Generate Submission

out_rows = []
for img_id, cands in all_masks.items():
    kept = subtract_overlaps([m for _, m in cands], MIN_AREA_FINAL)
    for idx, m in enumerate(kept, 1):
        out_rows.append((f'{img_id}_{idx}', encode(m)))

SUB = '/kaggle/working/submission_REFERENCE_039.csv'
with open(SUB, 'w', newline='') as f:
    w = csv.writer(f)
    w.writerow(['filament_id', 'segmentation_rle'])
    for fid, c in out_rows: w.writerow([fid, c])

counts = defaultdict(int)
for fid, _ in out_rows: counts[fid.rsplit('_',1)[0]] += 1
vals = list(counts.values())
print(f"Images: {len(counts)}/{len(test_files)}, rows: {len(out_rows)}, mean {np.mean(vals):.2f}/img")
print(f"Saved: {SUB}")

# Sanity check
with open(SUB) as f:
    rows_check = list(csv.DictReader(f))
by_img = defaultdict(list)
for r in rows_check:
    img_id = r['filament_id'].rsplit('_', 1)[0]
    rle = {'size': [2048, 2048], 'counts': r['segmentation_rle'].encode('ascii')}
    by_img[img_id].append(mu.decode(rle))
overlaps = 0
for img_id, masks in by_img.items():
    claimed = np.zeros((2048, 2048), np.uint8)
    for m in masks:
        if (claimed & m).any(): overlaps += 1
        claimed |= m
print(f"Overlaps: {overlaps}")

Images: 178/180, rows: 1104, mean 6.20/img
Saved: /kaggle/working/submission_REFERENCE_039.csv
Overlaps: 0
